# Fase 1a · Confirmar la tercera ley (quorum) — Mac mini

Plan de implementación v4, Fase 0 (verificación) + Fase 1a.

1. Comprueba entorno y hashes de los dos archivos copiados.
2. Verifica que `sin_colonias` sigue siendo idéntico bit a bit al protocolo guardado (**355/355**). Si falla, el notebook se detiene.
3. Corre `A2_sin_quorum` (sin colonias + sin quorum) a 50k y 500k, D10/D50, 15 corridas, semillas de la A3.

Ejecutar **celda por celda** o *Run All*: cada celda aborta si la anterior no pasó. Si se interrumpe, volver a ejecutar: `correr_marea2.py` retoma donde quedó.

**Solo en el Mac mini.** No ejecutar `experimento_MAREA2.ipynb`.

In [ ]:
import sys, os, hashlib, subprocess, opfunu

WORKERS = 20
ESPERADO = {
    "marea2.py":                 "4195810f8f95bb8792846cc6d92ac3fb8957f976df5658ce87d94b74d2a3fc17",
    "verificar_sin_colonias.py": "d1f7667ea3f4a92cc2b5fdc4d5801dbcd01fe7e66bf6c1675e96168a96961ea0",
}

def correr(*args):
    """Ejecuta un script con el mismo Python del kernel y muestra la salida en vivo."""
    p = subprocess.Popen([sys.executable, *args], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for linea in p.stdout:
        if "pkg_resources" not in linea:
            print(linea, end="")
    return p.wait()

print("python :", sys.executable)
print("carpeta:", os.getcwd())
print("opfunu :", opfunu.__version__)
assert opfunu.__version__ == "1.0.4", "opfunu tiene que ser 1.0.4 (1.0.1 esta roto)"
for f, h in ESPERADO.items():
    real = hashlib.sha256(open(f, "rb").read()).hexdigest()
    print(f"{f:28s} {real[:16]}  {'OK' if real == h else 'DISTINTO'}")
    assert real == h, f"{f} no es la version del commit 50447ba: copiala de nuevo"
print("\nEntorno y archivos OK")

## 1 · Verificar `sin_colonias` (unos minutos)
Tiene que decir **355/355 corridas identicas bit a bit**.

In [ ]:
rc = correr("verificar_sin_colonias.py", "--runs", "5", "--workers", str(WORKERS))
assert rc == 0, "sin_colonias NO es identico: no seguir, mandar la salida a Claude"
VERIFICADO = True
print("\nsin_colonias verificado: se puede poner el tag y seguir")

## 2 · Fase 1a a 50k (minutos)

In [ ]:
assert globals().get("VERIFICADO"), "primero la celda de verificacion"
rc = correr("correr_marea2.py", "--ablacion", "--cfgs", "A2_sin_quorum",
            "--maxfes", "50000", "--runs", "15", "--workers", str(WORKERS))
assert rc == 0

## 3 · Fase 1a a 500k (~3 h: ~1 h D10, ~2 h D50)

In [ ]:
assert globals().get("VERIFICADO"), "primero la celda de verificacion"
rc = correr("correr_marea2.py", "--ablacion", "--cfgs", "A2_sin_quorum",
            "--maxfes", "500000", "--runs", "15", "--workers", str(WORKERS))
assert rc == 0

## 4 · Comprobar los 4 JSON y traerlos al MacBook
Copiarlos a `resultados_experimento/ablacion/` del MacBook (no a la raíz).

In [ ]:
import json, glob, math
ABL = "resultados_experimento/ablacion"
for d in (10, 50):
    for fes in (50000, 500000):
        p = f"{ABL}/MAREA2_A2_sin_quorum_D{d}_FES{fes}.json"
        if not os.path.exists(p):
            print(f"FALTA  {p}"); continue
        v = json.load(open(p))
        malos = sum(1 for x in v.values() if not math.isfinite(x))
        print(f"{'OK   ' if len(v) == 71*15 and not malos else 'REVISAR'} {os.path.basename(p)}: "
              f"{len(v)}/{71*15} corridas, {malos} no finitos")